In [ ]:
import numpy as np
import numpy.random as npr
import matplotlib.pyplot as plt
import ssm
from neurodatatypes import Session
from ssm.util import find_permutation
from glm_hmm_utils import *
from pathlib import Path

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams.update({'font.size': 18})

FIGPATH = Path().home() / 'state_figs'

In [ ]:
# Set the parameters of the GLM-HMM
obs_dim = 1           # number of observed dimensions
num_categories = 2    # number of categories for output
DPATH = 'X:\Widefield'
MICE = ['mSM63','mSM64','mSM65','mSM66']
MICE = ['mSM65']
stim_vals = [-1, -.8, -.6, -.4, -.2, .2, .4, .6, .8, 1] # discrimination
#stim_vals = [-1, -.6, -.2, .2, .6,discrimination 1] #  (subsampled)
#stim_vals = [-1, 1] # detection

In [ ]:
gen_weights = np.array([[[4, .1, .2, .2]], [[.5, 2.5, .03, .5]], [[1.7, -1.8, 3, 1.5]]])
gen_log_trans_mat = np.log(np.array([[[0.98, 0.01, 0.01], [0.02, 0.95, 0.03], [0.01, 0.05, 0.94]]]))

#gen_weights = np.array([[[0, 4, 2, 0]], [[0, .2, .2, 5]], [[4, .1, .2, .2]], [[.5, 2.5, .03, .5]], [[1.7, -1.8, 3, 1.5]]])
#gen_log_trans_mat = np.log(np.array([[[0.94, 0.01, 0.01, .02, .02], [0.02, 0.88, 0.03, .06, .01], [0.01, 0.05, 0.93, .01, .02], [.04, .02, .1, .83, .01], [.01,.03,.04,.03,.89]]]))


gen_weights = np.array([[[2.5, .1]], [[1, 1.5]], [[1.2, -1.7]]])
gen_log_trans_mat = np.log(np.array([[[0.98, 0.01, 0.01], [0.05, 0.92, 0.03], [0.03, 0.06, 0.91]]]))

gen_weights = np.array([[[1.9, .1]], [[1, 1]], [[1.2, -1.1]]])
gen_log_trans_mat = np.log(np.array([[[0.98, 0.01, 0.01], [0.05, 0.92, 0.03], [0.03, 0.06, 0.91]]]))
#gen_weights = np.array([[[4, .5]], [[3, -2]], [[3, 2]], [[1, 1]]])
#gen_log_trans_mat = np.log(np.array([[[0.93, 0.01, 0.01, .05], [0.05, 0.92, 0.02, .01], [0.03, 0.03, 0.93, .01], [.05, .03, .03, .89]]]))

input_dim = gen_weights.shape[2]
num_states = gen_weights.shape[0]
# Make a GLM-HMM
true_glmhmm = ssm.HMM(num_states, obs_dim, input_dim, observations="input_driven_obs", 
                   observation_kwargs=dict(C=num_categories), transitions="standard")

#true_glmhmm = ssm.HMM(num_states,obs_dim, input_dim, observations="input_driven_obs", 
#                         observation_kwargs=dict(C=num_categories,prior_sigma=2),
#                         transitions="sticky", transition_kwargs=dict(alpha=3,kappa=0)
true_glmhmm.observations.params = gen_weights
true_glmhmm.transitions.params = gen_log_trans_mat

In [ ]:
state_labels = np.arange(1,num_states+1)
# Plot generative parameters:
fig = plt.figure(figsize=(10, 6), dpi=300, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
cols = ['blue','orange','red','green','pink']
for k in range(num_states):
    plt.plot(range(input_dim), gen_weights[k][0], marker='o',
             color=cols[k], linestyle='-',
             lw=1.5, label="state " + str(k+1))
plt.yticks(fontsize=10)
plt.ylabel("GLM weight", fontsize=15)
plt.xlabel("covariate", fontsize=15)
plt.xticks([0, 1], ['stimulus', 'bias'], fontsize=12, rotation=45)
plt.axhline(y=0, color="k", alpha=0.5, ls="--")
plt.legend()
plt.title("Generative weights", fontsize = 15)

plt.subplot(1, 2, 2)
gen_trans_mat = np.exp(gen_log_trans_mat)[0]
plt.imshow(gen_trans_mat, vmin=-0.8, vmax=1, cmap='bone')
for i in range(gen_trans_mat.shape[0]):
    for j in range(gen_trans_mat.shape[1]):
        text = plt.text(j, i, str(np.around(gen_trans_mat[i, j], decimals=2)), ha="center", va="center",
                        color="k", fontsize=12)
plt.xlim(-0.5, num_states - 0.5)
plt.xticks(range(0, num_states), state_labels, fontsize=10)
plt.yticks(range(0, num_states), state_labels, fontsize=10)
plt.ylim(num_states - 0.5, -0.5)
plt.ylabel("state t", fontsize = 15)
plt.xlabel("state t+1", fontsize = 15)
plt.title("Generative transition matrix", fontsize = 15)
plt.savefig('generative.pdf',format='pdf', dpi=300)

## Create external input sequences

In [ ]:
# simulated stimuli
#num_sess = 8 # number of example sessions
#num_trials_per_sess = 410 # number of trials in a session
#inpts = np.ones((num_sess, num_trials_per_sess, input_dim)) # initialize inpts array
#inpts[:,:,0] = np.random.choice(stim_vals, (num_sess, num_trials_per_sess)) # generate random sequence of stimuli
#inpts = list(inpts) #convert inpts to correct format

In [ ]:
#simulate choices from actual stimuli that were presented to mice
sessions = []
for mouse in MICE:
    #sessionlist = Session.get_sessions(DPATH, mouse, max_nochoice = 5, modality = 2, min_trials = 50, discrim_min = .77, discrim_max = 1, assisted_cutoff = .95, singlespout_cutoff = .02) # This should be all audio discriminatin sessions
    sessionlist = Session.get_sessions(DPATH, mouse, min_percent_correct=.68, max_nochoice=20, modality=2, min_trials=50, discrim_min=.5, discrim_max=1, assisted_cutoff=.9, singlespout_cutoff=.05)
    sessions.extend(sessionlist)
    print(f'There are {len(sessionlist)} sessions for mouse {mouse}')

inpts = GlmHmm(sessions, 'target_rate', input_terms_list=['coherence','bias','wsls','previous_choice']).inpts
inpts = GlmHmm(sessions, 'target_rate', input_terms_list=['coherence','bias',]).inpts

## Simulate states and choices

In [ ]:
np.concatenate(inpts).shape

In [ ]:
# Generate a sequence of latents and choices for each session
true_latents, true_choices = [], []
for sess in range(len(inpts)):
    true_z, true_y = true_glmhmm.sample(len(inpts[sess]), input=inpts[sess])
    true_latents.append(true_z)
    true_choices.append(true_y)
    
# Calculate true loglikelihood
true_ll = true_glmhmm.log_probability(true_choices, inputs=inpts) 
print("true ll = " + str(true_ll))

In [ ]:
#for s in inpts:
#    plt.plot(s[:,0])
#    plt.xlabel('Trial number')
#    plt.ylabel('Binned stimulus coherence')
#    plt.title('example session - starts with detection only trials')
#    plt.show()

## Fit the GLM-HMM

In [ ]:
N_iters = 10_000 # maximum number of EM iterations. Fitting with stop earlier if increase in LL is below tolerance specified by tolerance parameter

#new_glmhmm = ssm.HMM(num_states, obs_dim, input_dim, observations="input_driven_obs", 
                   #observation_kwargs=dict(C=num_categories), transitions="standard")

#fit_ll = new_glmhmm.fit(true_choices, inputs=inpts, method="em", num_iters=N_iters, tolerance=10**-6)


new_glmhmm = ssm.HMM(num_states, obs_dim, input_dim, observations="input_driven_obs", 
                         observation_kwargs=dict(C=num_categories,prior_sigma=.5),
                         transitions="sticky", transition_kwargs=dict(alpha=1,kappa=0))
fit_ll = new_glmhmm.fit(true_choices, inputs=inpts, method='em', num_iters=N_iters, tolerance=10**-6)


In [ ]:

# Plot the log probabilities of the true and fit models. Fit model final LL should be greater 
# than or equal to true LL.
fig = plt.figure(figsize=(4, 3), dpi=80, facecolor='w', edgecolor='k')
plt.plot(fit_ll, label="EM")
plt.plot([0, len(fit_ll)], true_ll * np.ones(2), ':k', label="True")
plt.legend(loc="lower right")
plt.xlabel("EM Iteration")
plt.xlim(0, len(fit_ll))
plt.ylabel("Log Probability")
plt.show()

In [ ]:
#print(len(new_glmhmm.most_likely_states(true_choices[0], input=inpts[0])))
#new_glmhmm.permute(find_permutation(true_latents[0], new_glmhmm.most_likely_states(true_choices[0], input=inpts[0])))
fig = plt.figure(figsize=(10, 6), dpi=300, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)

cols = ['blue','orange','red','green', 'pink']
recovered_weights = new_glmhmm.observations.params
for k in range(num_states):
    if k ==0:
        plt.plot(range(input_dim), gen_weights[k][0], marker='o',
                 color=cols[k], linestyle='-',
                 lw=1.5, label="generative")
        plt.plot(range(input_dim), recovered_weights[k][0], color=cols[k],
                     lw=1.5,  label = "recovered", linestyle = '--')
    else:
        plt.plot(range(input_dim), gen_weights[k][0], marker='o',
                 color=cols[k], linestyle='-',
                 lw=1.5, label="")
        plt.plot(range(input_dim), recovered_weights[k][0], color=cols[k],
                     lw=1.5,  label = '', linestyle = '--')
plt.yticks(fontsize=10)
plt.ylabel("GLM weight", fontsize=15)
plt.xlabel("covariate", fontsize=15)
plt.xticks([0, 1], ['stimulus', 'bias'], fontsize=12, rotation=45)
plt.axhline(y=0, color="k", alpha=0.5, ls="--")
plt.legend()
plt.title("Weight recovery", fontsize=15)

plt.subplot(1, 2, 2)
gen_trans_mat = np.exp(new_glmhmm.transitions.params)[0]
plt.imshow(gen_trans_mat, vmin=-0.8, vmax=1, cmap='bone')
for i in range(gen_trans_mat.shape[0]):
    for j in range(gen_trans_mat.shape[1]):
        text = plt.text(j, i, str(np.around(gen_trans_mat[i, j], decimals=2)), ha="center", va="center",
                        color="k", fontsize=12)
plt.xlim(-0.5, num_states - 0.5)
plt.xticks(range(0, num_states), state_labels, fontsize=10)
plt.yticks(range(0, num_states), state_labels, fontsize=10)
plt.ylim(num_states - 0.5, -0.5)
plt.ylabel("state t", fontsize = 15)
plt.xlabel("state t+1", fontsize = 15)
plt.title("Recovered transition matrix", fontsize = 15)
plt.savefig('recovered.pdf',format='pdf', dpi=300)

In [ ]:
fig = plt.figure(figsize=(5, 2.5), dpi=80, facecolor='w', edgecolor='k')
plt.subplot(1, 2, 1)
gen_trans_mat = np.exp(gen_log_trans_mat)[0]
plt.imshow(gen_trans_mat, vmin=-0.8, vmax=1, cmap='bone')
for i in range(gen_trans_mat.shape[0]):
    for j in range(gen_trans_mat.shape[1]):
        text = plt.text(j, i, str(np.around(gen_trans_mat[i, j], decimals=2)), ha="center", va="center",
                        color="k", fontsize=12)
plt.xlim(-0.5, num_states - 0.5)
plt.xticks(range(0, num_states), state_labels, fontsize=10)
plt.yticks(range(0, num_states), state_labels, fontsize=10)
plt.ylim(num_states - 0.5, -0.5)
plt.ylabel("state t", fontsize = 15)
plt.xlabel("state t+1", fontsize = 15)
plt.title("generative", fontsize = 15)


plt.subplot(1, 2, 2)
recovered_trans_mat = np.exp(new_glmhmm.transitions.log_Ps)
plt.imshow(recovered_trans_mat, vmin=-0.8, vmax=1, cmap='bone')
for i in range(recovered_trans_mat.shape[0]):
    for j in range(recovered_trans_mat.shape[1]):
        text = plt.text(j, i, str(np.around(recovered_trans_mat[i, j], decimals=2)), ha="center", va="center",
                        color="k", fontsize=12)
plt.xlim(-0.5, num_states - 0.5)
plt.xticks(range(0, num_states), state_labels, fontsize=10)
plt.yticks(range(0, num_states), state_labels, fontsize=10)
plt.ylim(num_states - 0.5, -0.5)
plt.title("recovered", fontsize = 15)
plt.subplots_adjust(0, 0, 1, 1)

In [ ]:
# Get expected states:
#sess_id = 25 #session id; can choose any index between 0 and num_sess-1

posterior_probs = [new_glmhmm.expected_states(data=data, input=inpt)[0]
                for data, inpt
                in zip(true_choices, inpts)]

for i in range(len(posterior_probs)):
    true_states = true_latents[i]
    fig = plt.figure(figsize=(5, 2.5), dpi=80, facecolor='w', edgecolor='k')
    for k in range(num_states):
        s = np.where(true_states == k)[0]
        bar = np.empty_like(s, dtype=float)
        bar[:] = 1.1
        plt.plot(s, bar, color=cols[k], linewidth = 8)

        plt.plot(posterior_probs[i][:, k], label="State " + str(k + 1), lw=2,
                 color=cols[k])
    #plt.ylim((-0.01, 2.01))
    plt.yticks([0, 0.5, 1], fontsize = 10)
    plt.xlabel("trial #", fontsize = 15)
    plt.ylabel("p(state)", fontsize = 15)